# Recurrent Neural NetworkによるBitcoinの価格予測

---
## 目的
Recurrent Neural Networkを使ってBitcoinの価格予測を行う．リカレントニューラルネットワークの構造や学習方法については`rnn_power.ipynb`で説明したものと同様のため，ここでは省略する．
また，CSVファイルなどのテキストファイルで整理・保存された数値データを扱うためのデータセットオブジェクトの作成を行う．


## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
import os
import csv
import zipfile
import gdown
import torch
import torch.nn as nn
from torch.utils.data import Dataset
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データのダウンロード

プログラムに必要なデータをダウンロードします．

今回使用するデータは，Kaggleで公開されているBitcoinの価格を予測するデータセットです．

https://www.kaggle.com/datasets/team-ai/bitcoin-price-prediction


In [ ]:
if not os.path.isdir('./BitcoinPricePrediction'):
    gdown.download('https://drive.google.com/uc?id=1_Gdneij6TP6CK_HCommCtbaitVfoY-fN', 'BitcoinPricePrediction.zip', quiet=False)
    with zipfile.ZipFile('BitcoinPricePrediction.zip') as f:
        f.extractall('./')

ここで，一度データセットを確認してみましょう．

データ（フォルダ）を確認すると，BitcoinPricePredictionフォルダの中にTraining.csvとTest.csvという二つのCSVファイルが保存されています．

### CSVファイルの中身

それぞれの中身を見ると，

* Date: 日付
* Open: 始値
* High: 最高値
* Low: 最安値
* Close: 終値
* Volume: 取引ボリューム（取引数量）
* Market Cap: 時価総額

という列があり，それぞれの日付で値を持っていることがわかります．

また，Dateの値を確認すると，新 --> 古の順番に日付が並んでいることがわかります．

今回は，「Open, High, Low, Close」の値から翌日の「Open, High, Low, Close」を予測する再帰型ニューラルネットワークを構築して学習してみましょう．

## データセットクラスの作成

ここでは，ダウンロードしたCSVファイルの形式に合わせて，PyTorchのデータセットクラスを自作します．

In [ ]:
class BitcoinPriceDataset(Dataset):

  def __init__(self, csv_file_path, time_window=10, max_price=None):
    super().__init__()

    self.csv_file_path = csv_file_path
    self.time_window = time_window

    ### csvファイルの読み込み
    with open(self.csv_file_path, 'r') as file:
      reader = csv.reader(file)

      ### 1行ずつデータをリストへ追加
      csv_data_source = []
      for row in reader:
        csv_data_source.append(row)

    ### ヘッダー行の保存（学習には使用しない）
    self.header = csv_data_source[0]
    ### データ行の保存（ヘッダ以外の行を保存）
    self.csv_data = csv_data_source[1:]

    ### 総データ数の保存
    self.num_data = len(self.csv_data)

    ### 日付データの保存（新旧の並び替えにのみ使用する）
    self.date = []
    for row in self.csv_data:
      self.date.append(row[0])

    ### 数値データの保存
    self.bitcoin_data = torch.zeros([self.num_data, 4], dtype=torch.float32)
    for i, row in enumerate(self.csv_data):
      self.bitcoin_data[i, 0] = float(row[1])
      self.bitcoin_data[i, 1] = float(row[2])
      self.bitcoin_data[i, 2] = float(row[3])
      self.bitcoin_data[i, 3] = float(row[4])

    ### データの順番を入れ替え（新~旧 --> 旧~新）
    self.date.reverse()
    self.bitcoin_data = torch.flipud(self.bitcoin_data)

    ### 最大の価格値
    if max_price is None:
      self.max_price = torch.max(self.bitcoin_data)
    else:
      self.max_price = max_price

    ### 数値データの正規化（0.0 ~ 1.0）
    self.bitcoin_data /= self.max_price

  def __getitem__(self, item):

    input = self.bitcoin_data[item:item+self.time_window, :]
    output = self.bitcoin_data[item+1:item+self.time_window+1, :]

    return input, output

  def __len__(self):
    return self.num_data - self.time_window

## ネットワークモデルの定義

続いてネットワークを定義します．

In [ ]:
class MyLSTM(nn.Module):

  ### ネットワーク構造
  # LSTM (LSTMCell) --> 全結合層 --> 予測結果

  def __init__(self, in_size=4, out_size=4, hidden_size=32):
    super().__init__()

    ### LSTM層の定義
    self.recurrent = nn.LSTMCell(input_size=in_size, hidden_size=hidden_size)

    ### 全結合層の定義
    self.fc = nn.Linear(in_features=hidden_size, out_features=out_size)

  def forward(self, x, hx, cx):
    hx, cx = self.recurrent(x, (hx, cx))
    h = self.fc(hx)
    return h, hx, cx

## 学習の準備

ここでは，学習に必要な

* ネットワークモデル
* 誤差関数
* 最適化手法
* データセット

の定義を行います．

**DataLoaderのnum_workersについて**

`torch.utils.data.DataLoader`の引数である`num_workers`は，データを読み込んで準備する処理を並列処理するための引数です．例えば，`num_workers=10`とした場合には，10並列でデータの読込処理 (データセットクラスの`__getitem__()`) を10並列で実行してくれます．そのため，使用する計算機のCPU性能に合わせて，ある程度大きな数を指定しておくとデータの読込処理が早くなり，学習の高速化が期待できます．

In [ ]:
### ネットワークモデル
n_hidden = 128
model = MyLSTM(in_size=4, out_size=4, hidden_size=n_hidden).to(device)

### 誤差関数
criterion = nn.MSELoss().to(device)

### 最適化関数
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

### データセットクラス
time_window = 10
train_dataset = BitcoinPriceDataset(csv_file_path="BitcoinPricePrediction/Training.csv", time_window=time_window, max_price=None)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=10, shuffle=True, num_workers=2)

### 学習データ内の最大の価格の値を保存 (後程テストに使用します)
train_max_price = train_dataset.max_price

## 学習の開始

上で定義したモデルや最適化手法，データセットを用いて学習を行います．
エポックごとにモデルのパラメータを`checkpoint-{epoch}.pt`として保存し，評価の際に読み込みます．

In [ ]:
### Epoch数などの指定
num_epochs = 50

### ネットワークを学習モードへ変更
model.train()

### 学習経過を保存するためのリストを用意
loss_list = []

### 学習ループ (for文)
for epoch in range(1, num_epochs+1):
  # 1 epochごとの学習経過を計算するための変数を用意
  loss_sum = 0.0

  for input, label in train_loader:
    input = input.to(device)
    label = label.to(device)

    # 隠れ状態の変数の初期化
    hx = torch.zeros(input.size(0), n_hidden, device=device)
    cx = torch.zeros(input.size(0), n_hidden, device=device)

    loss = 0.0
    for time_index in range(input.size(1)):
      y, hx, cx = model(input[:, time_index, :], hx, cx)
      loss += criterion(y, label[:, time_index, :])

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # lossを加算（学習経過確認用）
    loss_sum += loss.item()

  ### epochが終了したタイミングでそのエポックの平均誤差を表示（1 epoch内のiterationとtime_windowで割った値）
  print("Epoch:", epoch, ", Loss:", loss_sum / len(train_loader) / time_window)

  ### 上で表示した数値をリストへ保存
  loss_list.append(loss_sum / len(train_loader) / time_window)

## 評価

学習したモデルを用いて評価を行います．

### 推論と結果の表示

読み込んだ学習済みモデルを用いて予測を行います．

今回のネットワークは，数値データ（価格）の予測値を出力するモデルのため，誤差ではなく，その予測値をリストに保存して，後程グラフ表示をして結果を確認します．

#### 学習データに対する予測

まずは，学習に使用したデータでどの程度予測できるかを確認します．`rnn_power.ipynb`と同様に，「1ステップ先予測」と「自己回帰的な複数ステップ先予測」の2つの方法で評価を行います．


##### 1ステップ先予測
毎時刻，真のデータ（Open, High, Low, Close）をそのままモデルに入力して翌日のOpen, High, Low, Closeを予測します．これは，直前の状態が正確にわかっている前提での予測（教師強制; teacher forcing）であり，モデル自体の性能を確認するには有用ですが，遠い未来の価格を予測したいという実際の運用場面を反映したものではありません．

In [ ]:
pred = []
true = []

train_dataset_eval = BitcoinPriceDataset(csv_file_path="BitcoinPricePrediction/Training.csv", time_window=1, max_price=train_max_price)
train_loader_eval = torch.utils.data.DataLoader(train_dataset_eval, batch_size=1, shuffle=False, num_workers=2)

model.eval()

with torch.no_grad():

  # 隠れ状態の変数の初期化
  hx = torch.zeros(1, n_hidden, device=device)
  cx = torch.zeros(1, n_hidden, device=device)

  for input, label in train_loader_eval:
    input = input.to(device)
    label = label.to(device)

    output, hx, cx = model(input[:, 0, :], hx, cx)

    pred.append(output.tolist())
    true.append(label.tolist())

### 保存した正解・予測結果（リスト形式）をtorch.tensor形式に変換
pred_tensor = torch.tensor(pred).squeeze()
true_tensor = torch.tensor(true).squeeze()

### グラフの横軸用のリストを用意
time_index = list(range( pred_tensor.size(0) ))

### グラフの描画・表示・保存
plt.plot(time_index, pred_tensor[:, 0], '-', color='tab:blue', label='open pred')
plt.plot(time_index, true_tensor[:, 0], '--', color='tab:blue', label='open true')
plt.plot(time_index, pred_tensor[:, 1], '-', color='tab:orange', label='high pred')
plt.plot(time_index, true_tensor[:, 1], '--', color='tab:orange', label='high true')
plt.plot(time_index, pred_tensor[:, 2], '-', color='tab:green', label='low pred')
plt.plot(time_index, true_tensor[:, 2], '--', color='tab:green', label='low true')
plt.plot(time_index, pred_tensor[:, 3], '-', color='tab:red', label='close pred')
plt.plot(time_index, true_tensor[:, 3], '--', color='tab:red', label='close true')
plt.xlabel("day")
plt.ylabel("price")
plt.title("Prediction Results for Training Data (1-step)")
plt.legend()
plt.show()
plt.clf()

##### 自己回帰的な複数ステップ先予測
より現実的な評価として，最初の`warmup_steps`ステップだけ真のデータをそのまま入力して`hx`を準備し，その後の`pred_duration`ステップは真のOpen, High, Low, Closeの値を使わず，直前の自分自身の予測値をそのまま次の入力として使い回しながら予測を続けます．このモデルはOpen, High, Low, Closeのすべてを予測するため，ここでは`rnn_power.ipynb`や修正前のように一部の要素だけを真値のまま残す必要はなく，入力全体を自分自身の予測値に置き換えます．この自己回帰予測により，モデルが自身の予測誤差を蓄積させながら，遠い未来の価格をどれだけ正確に予測できるかを確認できます．

In [ ]:
warmup_steps = 250    # 最初に真のデータを入力してhxを準備するステップ数
pred_duration = 250  # その後，予測値を自己回帰的に入力し続けるステップ数

pred_ar = []
true_ar = []

hx_ar = torch.zeros(1, n_hidden, device=device)
cx_ar = torch.zeros(1, n_hidden, device=device)
prev_pred = None  # 自己回帰予測中に直前の自分自身の予測値（Open, High, Low, Close）を保持

with torch.no_grad():

  for t, (input, label) in enumerate(train_loader_eval):
    input = input.to(device)
    label = label.to(device)

    if t < warmup_steps:
      # 最初のwarmup_stepsステップは真のデータをそのまま入力し，hxを準備する
      input_ar = input[:, 0, :]
    else:
      # 以降は入力全体を直前の自分自身の予測値に置き換える
      input_ar = prev_pred

    output, hx_ar, cx_ar = model(input_ar, hx_ar, cx_ar)
    prev_pred = output.detach()

    pred_ar.append(output.tolist())
    true_ar.append(label.tolist())

    if t + 1 == warmup_steps + pred_duration:
      break

### 保存した正解・予測結果（リスト形式）をtorch.tensor形式に変換
pred_ar_tensor = torch.tensor(pred_ar).squeeze()
true_ar_tensor = torch.tensor(true_ar).squeeze()

### グラフの横軸用のリストを用意
time_index_ar = list(range( pred_ar_tensor.size(0) ))

### グラフの描画・表示・保存
plt.plot(time_index_ar, pred_ar_tensor[:, 0], '-', color='tab:blue', label='open pred')
plt.plot(time_index_ar, true_ar_tensor[:, 0], '--', color='tab:blue', label='open true')
plt.plot(time_index_ar, pred_ar_tensor[:, 1], '-', color='tab:orange', label='high pred')
plt.plot(time_index_ar, true_ar_tensor[:, 1], '--', color='tab:orange', label='high true')
plt.plot(time_index_ar, pred_ar_tensor[:, 2], '-', color='tab:green', label='low pred')
plt.plot(time_index_ar, true_ar_tensor[:, 2], '--', color='tab:green', label='low true')
plt.plot(time_index_ar, pred_ar_tensor[:, 3], '-', color='tab:red', label='close pred')
plt.plot(time_index_ar, true_ar_tensor[:, 3], '--', color='tab:red', label='close true')
plt.axvline(warmup_steps, color='gray', linestyle='--', label='warmup end')
plt.xlabel("day")
plt.ylabel("price")
plt.title("Prediction Results for Training Data (autoregressive, warmup={}, pred_duration={})".format(warmup_steps, pred_duration))
plt.legend()
plt.show()
plt.clf()

#### テストデータに対する予測

次に，テスト用データでどの程度予測できるかを確認します．`Test.csv`は学習データの直後の数日分（`Training.csv`最終日の翌日以降）しか含まれておらず，自己回帰的な複数ステップ先予測を試すには短すぎるため，ここでは1ステップ先予測のみを確認します．自己回帰的な複数ステップ先予測は，上の学習データを用いた評価で確認済みです．

In [ ]:
pred = []
true = []

test_dataset = BitcoinPriceDataset(csv_file_path="BitcoinPricePrediction/Test.csv", time_window=1, max_price=train_max_price)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)

model.eval()

with torch.no_grad():

  # 隠れ状態の変数の初期化
  hx = torch.zeros(1, n_hidden, device=device)
  cx = torch.zeros(1, n_hidden, device=device)

  for input, label in test_loader:
    input = input.to(device)
    label = label.to(device)

    output, hx, cx = model(input[:, 0, :], hx, cx)

    pred.append(output.tolist())
    true.append(label.tolist())


### 保存した正解・予測結果（リスト形式）をtorch.tensor形式に変換
pred_tensor = torch.tensor(pred).squeeze()
true_tensor = torch.tensor(true).squeeze()

### グラフの横軸用のリストを用意
time_index = list(range( pred_tensor.size(0) ))

### グラフの描画・表示・保存
plt.plot(time_index, pred_tensor[:, 0], '-', color='tab:blue', label='open pred')
plt.plot(time_index, true_tensor[:, 0], '--', color='tab:blue', label='open true')
plt.plot(time_index, pred_tensor[:, 1], '-', color='tab:orange', label='high pred')
plt.plot(time_index, true_tensor[:, 1], '--', color='tab:orange', label='high true')
plt.plot(time_index, pred_tensor[:, 2], '-', color='tab:green', label='low pred')
plt.plot(time_index, true_tensor[:, 2], '--', color='tab:green', label='low true')
plt.plot(time_index, pred_tensor[:, 3], '-', color='tab:red', label='close pred')
plt.plot(time_index, true_tensor[:, 3], '--', color='tab:red', label='close true')
plt.xlabel("day")
plt.ylabel("price")
plt.title("Prediction Results for Test Data (1-step)")
plt.legend()
plt.show()
plt.clf()

## 課題

1. `nn.LSTMCell`を`nn.RNNCell`や`nn.GRUCell`に変更して，予測精度がどのように変化するか確認してみましょう．
    * `LSTMCell`はセル状態`cx`も入出力している点に注意してください．
2. 予測対象を一部の値（例えば「Close」のみ）に絞った設定に変更し，予測精度や自己回帰予測の誤差の蓄積の様子がどのように変化するか比較してみましょう．
3. `time_window`や`n_hidden`などのハイパーパラメータを変更して，予測精度がどのように変化するか確認してみましょう．
4. `warmup_steps`や`pred_duration`の値を変えて，自己回帰的な複数ステップ先予測の誤差がどのように変化するか確認してみましょう．